Before running this notbook, please run codonaligner.ipynb and filting_introns.ipynb

In [6]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm

import trinuc_models as trinucs # this module must be in the same directory as this notebook

## CDS

In [7]:
#Setting up the rules for model fitting of cds regions
sm_noncds=trinucs.GNC_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_cds = [{"par_name": n, "is_independent": True} for n in paramnames]
GNC_subsmodel = get_app("model", "GNC_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_cds)

#Setting up the app for reading cds sequences
loader_cds = get_app("load_aligned", moltype="dna")   
omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

#create a concatenated alignment with all coding positions
cds_process = loader_cds + omit_degs_cds

In [8]:
import os

folder_in = paths.DATA_HUMCHIMPORANG115 + 'cds/chrm22/codon_aligned'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_cds = [r for r in cds_process.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

cds_alns = concat(nonconcat_cds)
cds_alns.source = "cds_alignments"

result_cds = GNC_subsmodel(cds_alns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_cds.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_cds))

result_cds.lf

   0%|          |00:00<?

   0%|          |00:00<?

GNC_CpG_ss
log-likelihood = -797964.5055
number of free parameters = 102
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.01               4.16    1.09    4.06    0.68
Orangutan     root        0.05               4.25    1.09    4.04    0.52
Chimpanzee    root        0.02               3.59    1.13    1.85    0.93
-------------------------------------------------------------------------

continued: 
=====================================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C    omega
---------------------------------------------------------------------
1.20    1.40    4.58    3.81    1.31    1.20    0.68    2.69     0.33
0.70    1.19    2.64    2.27    1.16    0.76    0.63    4.39     0.24
0.83    1.14    1.86    1.62    1.48    1.01    0.76    1.89     0.50
---------------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.02    0.04    0.01    0.01    0.02    0.01    0.01    0.01    0.03
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.00    0.02    0.02    0.01    0.01    0.02    0.04    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.03    0.01    0.02    0.01    0.02    0.02    0.00    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.05    0.01    0.02    0.03    0.05    0.02    0.01    0.04    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAC     TAT
----------------------------------------------------------------------------
0.01    0.03    0.02    0.01    0.00    0.02    0.03    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TCA     TCC     TCG     TCT     TGC     TGG     TGT     TTA     TTC     TTG
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
====
 TTT
----
0.01
----

## Intergenic Ancestral Repeats (IGAR) with codon substitution model

In [11]:
#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs = get_app("omit_degenerates", moltype="dna", motif_length=3)

#trim_stop_codons only remove codons at the end of the sequence, I need to make a custome app that removes stop codons instead
trim_stops = get_app("trim_stop_codons")

concat = get_app("concat", moltype="dna")

noncds_usingcodon_app = loader + rename_noncds + omit_degs + trim_stops

folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGARusingcodon = [r for r in noncds_usingcodon_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGARusingcodon = concat(nonconcat_IGARusingcodon)
alns_IGARusingcodon.source = "IGAR_alignments_usingcodon"

result_IGAR_usingcodon = GNC_subsmodel(alns_IGARusingcodon)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_IGAR_usingcodonmodel.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR_usingcodon))

result_IGAR_usingcodon.lf

   0%|          |00:00<?

AttributeError: 'NotCompleted' object has no attribute 'lf'

In [12]:
result_IGAR_usingcodon

NotCompleted(type=ERROR, origin=model, source="IGAR_alignments_usingcodon", message="Traceback (most recent call last):
  File "/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/evolve/likelihood_tree.py", line 261, in make_likelihood_tree_leaf
    likelihoods = get_matched_array(alphabet, moltype, uniq_motifs, dtype=float)
  File "/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/evolve/likelihood_tree.py", line 218, in get_matched_array
    for motif in moltype.resolve_ambiguity(ambig_motif, alphabet=alphabet):
                 ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/core/moltype.py", line 1112, in resolve_ambiguity
    raise c3_alphabet.AlphabetError(ambig_motif)
cogent3.core.alphabet.AlphabetError: TAG

The above exception was the direct cause of the following exception:

Traceback (most recent call last):


# Non cds

In [13]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_noncds)

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [16]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGAR = concat(nonconcat_IGAR)
alns_IGAR.source = "IGAR_alignments"

result_IGAR = GT_subsmodel(alns_IGAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_IGAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR))

result_IGAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -232050.4620
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        3.23              50.00    10.59    11.32    11.77
Orangutan     root        3.76              32.06     3.47     3.58     4.18
Chimpanzee    root        3.20              50.00    11.54    12.43    12.15
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.11    1.77    0.00    0.00    0.00    0.07
0.69    1.19    0.24    1.09    0.93    0.27    0.60    0.91
0.00    0.00    0.14    1.66    0.00    0.00    0.00    0.42
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.74    0.00    0.02    0.01    0.00    0.00    0.00    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.01    0.00    0.01    0.00    0.01    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.00    0.00    0.00
----------------------------

## Introns

In [17]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm22/noUTRs/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_introns = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_introns = concat(nonconcat_introns)
alns_introns.source = "introns_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_introns = GT_subsmodel(alns_introns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_introns.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_introns))

result_introns.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -88749.9393
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.65              17.82    1.02    0.94    0.66
Orangutan     root        2.60              10.89    0.95    1.01    1.14
Chimpanzee    root        0.63              18.84    1.27    1.16    1.07
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.55    0.62    0.40    0.44    0.83    0.51    0.71    0.83
0.62    0.90    0.76    0.69    0.89    0.68    1.16    1.04
0.76    0.84    0.54    0.53    0.94    0.61    1.09    1.22
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.01    0.02    0.02    0.01    0.01    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.01    0.02    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.03    0.01    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.01    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.02    0.02    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------

## 5' UTR

In [18]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm22/5UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_5UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_5UTR = concat(nonconcat_5UTR)
alns_5UTR.source = "5UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_5UTR = GT_subsmodel(alns_5UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_5UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_5UTR))

result_5UTR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -69460.0206
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.39              10.29    0.75    1.01    1.01
Orangutan     root        1.41               7.32    0.97    1.02    0.84
Chimpanzee    root        0.47               9.06    0.92    0.93    0.93
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.01    1.19    0.91    0.94    1.24    0.82    1.15    0.83
0.80    1.17    0.75    0.84    1.06    0.87    0.91    1.02
0.87    0.94    0.92    0.78    0.88    1.07    1.16    0.90
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.01    0.02    0.02    0.02    0.01    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.02    0.02    0.00    0.01    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.01    0.01    0.01    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.02    0.01    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------

## 3' UTR

In [19]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm22/3UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_3UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_3UTR = concat(nonconcat_3UTR)
alns_3UTR.source = "3UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_3UTR = GT_subsmodel(alns_3UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_3UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_3UTR))

result_3UTR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -48125.0449
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.56              16.78    0.62    0.64    0.26
Orangutan     root        2.73               8.63    0.87    0.84    0.70
Chimpanzee    root        0.78               9.59    0.76    0.90    0.77
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.63    0.73    0.32    0.42    0.71    0.46    0.60    0.65
0.68    1.25    0.73    0.64    1.24    0.77    0.78    1.05
1.01    1.37    0.74    0.89    0.96    0.65    1.27    1.08
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.01    0.02    0.01    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.01    0.01    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.03    0.03    0.00    0.02    0.02    0.01    0.01    0.03
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.02    0.01    0.01    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.01    0.00    0.02    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.02    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.02
----------------------------

## Introns Ancestral Repeats (IntronAR)

In [20]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intronsAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_intronsAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_intronsAR = concat(nonconcat_intronsAR)
alns_intronsAR.source = "intronsAR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_intronsAR = GT_subsmodel(alns_intronsAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_intronsAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_intronsAR))

result_intronsAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -95628.6695
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        2.12              50.00    33.49    35.21    26.46
Orangutan     root        3.75              12.14     2.16     2.05     2.73
Chimpanzee    root        2.03              50.00    30.57    31.96    30.32
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.07    0.06    2.70    0.29    0.00    0.00    0.26
0.93    1.25    0.47    1.18    0.88    0.53    1.26    1.12
0.00    0.00    0.04    3.15    0.00    0.00    0.00    0.00
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.46    0.01    0.03    0.01    0.01    0.01    0.00    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.02    0.01    0.00    0.01    0.02    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.02    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.00    0.00    0.01    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.00    0.01    0.01
----------------------------

## Intergenic distal (5kb distal from transcript)

In [22]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'distalIG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_distalIG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_distalIG = concat(nonconcat_distalIG)
alns_distalIG.source = "distalIG_alignments"

result_distalIG = GT_subsmodel(alns_distalIG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_distalIG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_distalIG))

result_distalIG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -110010.0640
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        3.17              50.00    10.07    10.83    10.92
Orangutan     root        3.48              50.00     5.77     6.07     6.58
Chimpanzee    root        3.07              50.00    12.47    13.43    13.43
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.10    1.72    0.00    0.00    0.00    0.00
0.75    1.04    0.19    1.20    0.86    0.32    0.30    0.98
0.00    0.00    0.14    2.02    0.00    0.00    0.00    0.31
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.75    0.00    0.02    0.01    0.00    0.00    0.01    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.01    0.00    0.00    0.00    0.01    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.00    0.00    0.00
----------------------------

## Intergenic proximal 5' (5kb proximal from transcript start)

In [23]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'proximal5IG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal5IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal5IG = concat(nonconcat_proximal5IG)
alns_proximal5IG.source = "proximal5IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal5IG = GT_subsmodel(alns_proximal5IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal5IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal5IG))

result_proximal5IG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -93891.8093
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        3.21              50.00     7.67     8.11     9.01
Orangutan     root        4.04              24.05     1.69     1.80     2.30
Chimpanzee    root        3.28              50.00    10.67    11.99    10.41
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.08    0.13    1.29    0.14    0.00    0.00    0.37
0.45    0.83    0.25    0.84    0.69    0.18    0.44    0.68
0.00    0.00    0.16    1.52    0.03    0.00    0.00    0.62
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.72    0.00    0.02    0.01    0.00    0.00    0.01    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.00    0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.00    0.01    0.01    0.00    0.00    0.01    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.00    0.00    0.00
----------------------------

## Intergenic proximal 3' (5kb proximal from transcript end)

In [24]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'proximal3IG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal3IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal3IG = concat(nonconcat_proximal3IG)
alns_proximal3IG.source = "proximal3IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal3IG = GT_subsmodel(alns_proximal3IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal3IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal3IG))

result_proximal3IG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -56833.8265
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        3.56              50.00    0.00    0.02    0.00
Orangutan     root        4.05              48.56    0.02    0.36    0.32
Chimpanzee    root        3.66              50.00    0.05    0.08    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.15    0.03    0.00    0.00    1.08    1.13
0.03    0.20    0.22    0.05    0.25    0.28    1.28    1.20
0.00    0.00    0.17    0.02    0.02    0.00    1.07    1.22
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.00    0.01    0.01    0.00    0.00    0.01    0.01    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.00    0.02    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.00    0.01    0.02    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.00    0.00    0.82
----------------------------